In [33]:
import os
import re
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain.prompts import ChatPromptTemplate
import pandas as pd

metadata_filtered_df = pd.read_csv("../../data/preprocessed/amazon-All_Beauty/metadata_filtered_df.csv")

In [34]:
import re 

def create_source_text(product):
        """Concatenate product information into a text string

        Args:
            product (dict): dictionary containing product information

        Returns:
            str: concatenated product information
        """
        description = "*" if product["description"] == "" else f"Description: {product['description']}"
        features = "*" if product["features"] == "" else f"Features: {product['features']}"
        details = "*" if product["details"] == "" else f"Details: {product['details']}"
        store = "*" if product["store"] == "" else f"Store: {product['store']}"
        categories = "*" if product["categories"] == "" else f"Categories: {product['categories']}"
        price = "*" if product["price"] == "" else f"Price: {product['price']}"
        author = "*" if product["author"] == "" else f"Author: {product['author']}"

        # concatenated_text = f"Parent ASIN: {product['parent_asin']}; Title: {product['title']}; Author: {author}; Description: {description}; Features: {features} - {details}; Store: {store}; Categories: {categories}; Price: {price};"
        concatenated_text = f"Parent ASIN: {product['parent_asin']}; Title: {product['title']}Description: {description};- {details}; Store: {store};"
        #print("Concatenated text:", concatenated_text)  # Debug: Print the output text
        return concatenated_text
    
def clean_string(string):
            string = re.sub(r'\[', '', string)
            string = re.sub(r'\]', '', string)
            string = re.sub(r'"', '', string)
            string = re.sub(r'\s+', ' ', string)
            string = re.sub("{", "", string)
            string = re.sub("}", "", string)
            return string

def clean_column(df, column_name):
    """
    Clean a column in the dataframe by applying clean_string to each element.
    Handles lists by converting them to strings first.
    
    Args:
        df (pandas.DataFrame): The dataframe containing the column to clean
        column_name (str): The name of the column to clean
        
    Returns:
        pandas.Series: The cleaned column
    """
    def safe_clean(value):
        if isinstance(value, list):
            # Convert list to string before cleaning
            return clean_string(str(value))
        elif pd.isna(value) or value is None:
            return ""
        else:
            return clean_string(str(value))
    
    return df[column_name].apply(safe_clean)

# Clean text columns that might contain lists
for column in ['details', 'features', 'categories', 'description', 'bought_together']:
    if column in metadata_filtered_df.columns:
        metadata_filtered_df[column] = clean_column(metadata_filtered_df, column)

metadata_filtered_df['source_text'] = metadata_filtered_df.apply(create_source_text, axis=1)


In [43]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
def extract_triplets(text):
    triplets = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    for token in text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
    return triplets

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")
gen_kwargs = {
    "max_length": 256,
    "length_penalty": 0,
    "num_beams": 3,
    "num_return_sequences": 3,
}

triples = []

def generate_triples(model,tokenizer,row):
    texts = [row.source_text]
    parent_asin = row.parent_asin
    # Tokenizer text
    model_inputs = tokenizer(texts, max_length=512, padding=True, truncation=True, return_tensors='pt')
    generated_tokens = model.generate(
        model_inputs["input_ids"].to(model.device),
        attention_mask=model_inputs["attention_mask"].to(model.device),
        **gen_kwargs
    )
    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)
    for idx, sentence in enumerate(decoded_preds):
        et = extract_triplets(sentence)
        for t in et:
            # link product to head / hyponym
            triples.append((parent_asin, t['type'], t['head']))
            # link head / hyponym to tail / hypernym
            triples.append((t['head'], t['type'], t['tail']))

for i in tqdm(range(0, len(metadata_filtered_df))):
  
    generate_triples(model,tokenizer,metadata_filtered_df.iloc[i])

distinct_triples = list(set(triples))





100%|██████████| 324/324 [12:40<00:00,  2.35s/it]


In [41]:
class NERPatternExtractor:
    def __init__(self,model_name,tokenizer_name):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def extract_triplets(text):
        triplets = []
        #TODO: change triplets list to dataframe
        relation, subject, relation, object_ = '', '', '', ''
        text = text.strip()
        current = 'x'
        for token in text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").split():
            if token == "<triplet>":
                current = 't'
                if relation != '':
                    triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
                    relation = ''
                subject = ''
            elif token == "<subj>":
                current = 's'
                if relation != '':
                    triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
                object_ = ''
            elif token == "<obj>":
                current = 'o'
                relation = ''
            else:
                if current == 't':
                    subject += ' ' + token
                elif current == 's':
                    object_ += ' ' + token
                elif current == 'o':
                    relation += ' ' + token
        if subject != '' and relation != '' and object_ != '':
            triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
        return triplets

    def generate_triples(self,row, triples_df):
        texts = [row.source_text]
        parent_asin = row.parent_asin
        # Tokenizer text
        model_inputs = self.tokenizer(texts, max_length=512, padding=True, truncation=True, return_tensors='pt')
        generated_tokens = self.model.generate(
            model_inputs["input_ids"].to(self.model.device),
            attention_mask=model_inputs["attention_mask"].to(self.model.device),
            **gen_kwargs
        )
        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)
        for idx, sentence in enumerate(decoded_preds):
            et = extract_triplets(sentence)
            for t in et:
                # link product to head / hyponym
                triples_df = pd.concat([triples_df,pd.DataFrame([{"head":parent_asin,"relation": t['type'], "tail": t['head']}])], ignore_index=True)
                # link head / hyponym to tail / hypernym
                triples_df = pd.concat([triples_df,pd.DataFrame([{"head":t['head'],"relation": t['type'], "tail": t['tail']}])], ignore_index=True)
        return triples_df
if __name__ == "__main__":
    DATASETS = ["Books", "All_Beauty", "Beauty_and_Personal_Care"]
    DATASET = DATASETS[1]
    DIR_NAME = "amazon-" # else: Last_fm, MovieLens

    NER_MODEL = "Babelscape/rebel-large"
    
    # get data
    current_dir = os.getcwd()
    data_path = os.path.join(current_dir, 'data', 'preprocessed', f'{DIR_NAME}{DATASET}', 'metadata_filtered_df.csv')
    metadata_filtered_df = pd.read_csv(data_path)


    # build triples dataframe
    triples_df = pd.DataFrame(columns=["head","relation", "tail"])

    ner_pattern_extractor = NERPatternExtractor(model_name=NER_MODEL,tokenizer_name=NER_MODEL)
    for i in tqdm(range(0, len(metadata_filtered_df))):
        triples_df = ner_pattern_extractor.generate_triples(metadata_filtered_df.iloc[i],triples_df)

    # drop duplicates
    triples_df = triples_df.drop_duplicates()

    # save triples to csv
    out_path = os.path.join(current_dir, 'data', 'relations', f'{DIR_NAME}{DATASET}/ner_relations.csv')
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    triples_df.to_csv(out_path, index=False)
    print(f"Saved {len(triples_df)} relations to {out_path}")

[('B081ZN3TD5', 'instance of', 'JPNK'),
 ('Spray Bottles', 'use', 'Spray'),
 ('B089CSF11Y', 'use', 'Spray Bottles'),
 ('JPNK 4PCS Anti-Static Detangling Fine & Wide Tooth Shower Comb Set',
  'manufacturer',
  'JPNK'),
 ('B089CSF11Y', 'instance of', 'Cherry'),
 ('JPNK', 'instance of', 'Brand'),
 ('B08LYT4Q2X', 'subclass of', 'Moisturizing'),
 ('B081ZN3TD5', 'manufacturer', 'JPNK 4PCS'),
 ('B08LYT4Q2X', 'part of', 'Moisturizing'),
 ('Moisturizing', 'subclass of', 'Body Oil'),
 ('JPNK 4PCS', 'manufacturer', 'JPNK'),
 ('Cherry', 'instance of', 'Brand'),
 ('B081ZN3TD5',
  'manufacturer',
  'JPNK 4PCS Anti-Static Detangling Fine & Wide Tooth Shower Comb Set'),
 ('Moisturizing', 'part of', 'Skin'),
 ('Moisturizing', 'subclass of', 'Oil')]

hearst_patterns 

In [32]:
class LLMHeirarchy:
    def __init__(self,model):
        self.model = OllamaLLM(model=model,temperature=0) # TODO: in AWS Sagemaker call model via endpoint, api-key

    def chain(self,prompt_type:str,input:str): 

        if prompt_type == "hearst":
            input_message = """
            Create a Hearst pattern for the following text:
            {Text}
            """

            system_prompt = """You are a helpful assistant that can construct Hearst patterns to extract entities and relationships from text.
            You will then need to return the Hearst patterns in a structured format.
            For entities you can use product groups, categories, features, store, ingredients, price range or other suitable entities.
            include not obvious relationships
            if you have information about the product, which are not in the text, you can include them in the hypernym.

            Rules:
            - You will only return the Hearst patterns, dont add notes.
            - Dont include parent_asin, titles in the patterns
            - Build patterns with A -> B, where A is the hypernym and B is the hyponym and B -> C, where B is the hypernym and C is the hyponym
            - Return the Hearst patterns in a structured format. without any regex.
            - Use the commom Hearst Patterns.
            - The output schema is: hypernym("entity"; "hyponym")
            - only use ; to separate the entities.

            Example:
            #1 input: "Harry Potter, The Lord of the Rings, The Prophet, The Alchemist or other books are bestsellers"
            output: hyponym("Harry Potter"; "book"), hyponym("The Lord of the Rings"; "book"), hyponym("The Prophet"; "book"), hyponym("The Alchemist"; "book")

            #2 input: "Harry Potter, The Lord of the Rings, The Prophet, The Alchemist or other books are bestsellers"
            output: hyponym("Harry Potter"; "book"), hyponym("The Lord of the Rings"; "book"), hyponym("The Prophet"; "book"), hyponym("The Alchemist"; "book")

            #3 input: "The book Harry Potter is a fantasy book with a lot of magic especially for teenagers"
            output: hyponym("book"; "Harry Potter"), hyponym("book"; "fantasy book"), hyponym("audience group"; "teenagers")

            #4 
            input: "Most New York Times bestsellers, especially Rich dad poor dad, The 48 laws of power or Atomic habits are self-help books"
            output: hyponym("New York Times bestsellers"; "Rich dad poor dad"), hyponym("New York Times bestsellers"; "The 48 laws of power"), hyponym("New York Times bestsellers"; "Atomic habits")

            #5
            input: "All horror movies are scary including the movie The Conjuring, Saw and The Exorcist"
            output: hyponym("horror movies"; "The Conjuring"), hyponym("horror movies"; "Saw"), hyponym("horror movies"; "The Exorcist")

            #6 
            input: "Babo Botanicals Sheer Zinc Continuous Spray Sunscreen SPF 30 is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
            output: hyponym("Aloe Vera"; "Ingredient"), hyponym("Sunflower Oil"; "Ingredient"), hyponym("Baby Skin"; "Effect"), hyponym("Kids Skin"; "Effect"), hyponym("Sensitive Skin"; "Effect"), hyponym("Vegan Product"; "Sustainable Product"),hyponym("Sustainable Product"; "Product"), hyponym("Mineral Active"; "Feature"), hyponym("Water-Resistant Product"; "Feature")

            #7 
            input: Hypernym: Glycerin Product, Hyponym: Cathy Doll L-Glutathione Magic Cream SPF 50 Whitening Sunscreen 138ml with Glycerin and whitening effect
            output hyponym("Glycerin Product": "Feature"), hyponym("Cathy Doll L-Glutathione Magic Cream SPF 50 Whitening Sunscreen": "Product"), hyponym("Glycerin": "Ingredient"), hyponym("Whitening effect": "Feature")

            #8
            input: "The product is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
            output: hyponym("Aloe Vera"; "Ingredient"), hyponym("Sunflower Oil"; "Ingredient"), hyponym("Baby Skin"; "Effect"), hyponym("Kids Skin"; "Effect"), hyponym("Sensitive Skin"; "Effect"), hyponym("Vegan Product"; "Product"), hyponym("Mineral Active"; "Feature"), hyponym("Water-Resistant Product"; "Feature")

            """

        elif prompt_type == "hierarchical":
            input_message = """
            Create a hierarchical topic model for the following text:
            {Text}
            """
            system_prompt = """
            Role:You are a helpful assistant that extract entities and relationships from text.
            Goal:Your goal is to find entities, relationships from the text.
            The entities should be able to be hierarchically related to each other.
            Make the entities as general as possible, so that they can be used for a hierarchical topic model and found in various contexts.
            You are allowed to use your knowledge about the product, if it is not in the text.
            
            Rules:
            - You will only return the Entities and relationships, dont add notes.
            - Dont include identifiers like parent_asin, titles in the patterns
            - Build patterns with A -> B, where A is the hypernym and B is the hyponym and B -> C, where B is the hypernym and C is the hyponym
            - Return without any regex.
            - The output schema is: "entity-1"("relation"; "entity-2")
            - only use ; to separate the entities.
            - Write all words in lowercase.
            - Use short words for entities and relationships.

            Example:
            #1input: "The product is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
            output: hyponym("product"; "sustainble ingredients"), hyponym("sustainble ingredients"; "aloe vera"), hyponym("sustainble ingredients"; "sunflower oil"), hyponym("Feautures"; "water resistant"), hyponym("Feautures"; "vegan"), hyponym("Feautures"; "mineral active"), hyponym("target audience"; "sensitive skin"), hyponym("target audience"; "kids skin"), hyponym("target audience"; "vegan")

            #2
            input: "The book Harry Potter is a fantasy book with a lot of magic especially for teenagers"
            output: hyponym("Harry Potter"; "fantasy book"), hyponym("fantasy book"; "book"), hyponym("Harry Potter"; "teenagers"), hyponym("Harry Potter"; "j.k. rowling")
            """
        prompt = ChatPromptTemplate(messages=[
    ("system", system_prompt),
    ("user", input_message)
    ], 
            input_variables=["Text"]
        )

        chain = prompt | self.model
        result = chain.invoke({f"Text": input})
        return result
    
    def create_triples(self,df,hypernym,hyponym,parent_asin): 
        df = pd.concat([df,pd.DataFrame([{"head":parent_asin,"relation": "related to", "tail": hyponym}])], ignore_index=True)
        df = pd.concat([df,pd.DataFrame([{"head":hyponym,"relation": "related to", "tail": hypernym}])], ignore_index=True)
        return df


    def post_process(self,result, df, row):
        """
        Extract the hypernym and hyponym from the result and build triples 

        Args:
            result (str): The answer hypernym and hyponym from the LLM
            df (pd.DataFrame): The dataframe to store the triples
            row (pd.Series): The row from the metadata_filtered_df, indicates the current product

        Returns:
            df (pd.DataFrame): The dataframe with the triples
        """
        parent_asin = row.parent_asin
        for line in result.split("\n"):
            if "hyponym" in line:
                match = re.search(r'hyponym\("([^"]+)"\s*;\s*"([^"]+)"\)', line) 
                if match:
                    hypernym, hyponym = match.groups()
                    df = self.create_triples(df,hypernym,hyponym,parent_asin)
        return df
    

if __name__ == "__main__":
    DATASETS = ["Books", "All_Beauty", "Beauty_and_Personal_Care"]
    DATASET = DATASETS[1]
    DIR_NAME = "amazon-" # else: Last_fm, MovieLens
    LLM_MODEL = "llama3.2"
    
    # Get the project root directory (Masterarbeit-Playground folder)
    current_dir = os.getcwd()
    data_path = os.path.join(current_dir, 'data', 'preprocessed', f'{DIR_NAME}{DATASET}', 'metadata_filtered_df.csv')
    metadata_filtered_df = pd.read_csv(data_path)

    # build triples dataframe
    triples_df = pd.DataFrame(columns=["head","relation", "tail"])

    # initialize llm hierarchy
    llm_hierarchy = LLMHeirarchy(model=LLM_MODEL)

# try with iloc 1:10
    for i,row in metadata_filtered_df.iloc[0:3].iterrows():
        result = llm_hierarchy.chain("hierarchical", input=row.source_text)
        print(result)
        print(f"Processing row {i+1} of {len(metadata_filtered_df)}")
        print("--------------------------------")
        
        triples_df = llm_hierarchy.post_process(result,triples_df,row)

    out_path = os.path.join(current_dir, 'data', 'relations', f'{DIR_NAME}{DATASET}','{LLM_MODEL}_triples.csv')
    triples_df.to_csv(out_path, index=False)
 


FileNotFoundError: [Errno 2] No such file or directory: '/Users/U725801/Documents/GitHub/Masterarbeit-Playground/src/playground/data/preprocessed/amazon-All_Beauty/metadata_filtered_df.csv'

In [16]:



input_message = """
Create a Hearst pattern for the following text:
{Text}
"""

system_prompt = """You are a helpful assistant that can construct Hearst patterns to extract entities and relationships from text.
You will then need to return the Hearst patterns in a structured format.
For entities you can use product groups, categories, features, store, ingredients, price range or other suitable entities.
include not obvious relationships
if you have information about the product, which are not in the text, you can include them in the hypernym.

Rules:
- You will only return the Hearst patterns, dont add notes.
- Dont include parent_asin, titles in the patterns
- Build patterns with A -> B, where A is the hypernym and B is the hyponym and B -> C, where B is the hypernym and C is the hyponym
- Return the Hearst patterns in a structured format. without any regex.
- Use the commom Hearst Patterns.
- The output schema is: hypernym("entity"; "hyponym")
- only use ; to separate the entities.

Example:
#1 input: "Harry Potter, The Lord of the Rings, The Prophet, The Alchemist or other books are bestsellers"
output: hyponym("Harry Potter"; "book"), hyponym("The Lord of the Rings"; "book"), hyponym("The Prophet"; "book"), hyponym("The Alchemist"; "book")

#2 input: "Harry Potter, The Lord of the Rings, The Prophet, The Alchemist or other books are bestsellers"
output: hyponym("Harry Potter"; "book"), hyponym("The Lord of the Rings"; "book"), hyponym("The Prophet"; "book"), hyponym("The Alchemist"; "book")

#3 input: "The book Harry Potter is a fantasy book with a lot of magic especially for teenagers"
output: hyponym("book"; "Harry Potter"), hyponym("book"; "fantasy book"), hyponym("audience group"; "teenagers")

#4 
input: "Most New York Times bestsellers, especially Rich dad poor dad, The 48 laws of power or Atomic habits are self-help books"
output: hyponym("New York Times bestsellers"; "Rich dad poor dad"), hyponym("New York Times bestsellers"; "The 48 laws of power"), hyponym("New York Times bestsellers"; "Atomic habits")

#5
input: "All horror movies are scary including the movie The Conjuring, Saw and The Exorcist"
output: hyponym("horror movies"; "The Conjuring"), hyponym("horror movies"; "Saw"), hyponym("horror movies"; "The Exorcist")

#6 
input: "Babo Botanicals Sheer Zinc Continuous Spray Sunscreen SPF 30 is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
output: hyponym("Aloe Vera"; "Ingredient"), hyponym("Sunflower Oil"; "Ingredient"), hyponym("Baby Skin"; "Effect"), hyponym("Kids Skin"; "Effect"), hyponym("Sensitive Skin"; "Effect"), hyponym("Vegan Product"; "Sustainable Product"),hyponym("Sustainable Product"; "Product"), hyponym("Mineral Active"; "Feature"), hyponym("Water-Resistant Product"; "Feature")

#7 
input: Hypernym: Glycerin Product, Hyponym: Cathy Doll L-Glutathione Magic Cream SPF 50 Whitening Sunscreen 138ml with Glycerin and whitening effect
output hyponym("Glycerin Product": "Feature"), hyponym("Cathy Doll L-Glutathione Magic Cream SPF 50 Whitening Sunscreen": "Product"), hyponym("Glycerin": "Ingredient"), hyponym("Whitening effect": "Feature")

#8
input: "The product is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
output: hyponym("Aloe Vera"; "Ingredient"), hyponym("Sunflower Oil"; "Ingredient"), hyponym("Baby Skin"; "Effect"), hyponym("Kids Skin"; "Effect"), hyponym("Sensitive Skin"; "Effect"), hyponym("Vegan Product"; "Product"), hyponym("Mineral Active"; "Feature"), hyponym("Water-Resistant Product"; "Feature")

"""

prompt = ChatPromptTemplate(messages=[
    ("system", system_prompt),
    ("user", input_message)
], 
    input_variables=["Text"]
)



In [18]:
import pandas as pd
model = OllamaLLM(model="llama3.2",temperature=0)
hearst_df = pd.DataFrame(columns=["parent_asin","hypernym", "hyponym"])

def chain(prompt, model, input): 
    chain = prompt | model
    result = chain.invoke({f"Text": input})
    return result

# initialize dataframe
def store_results(hypernym, hyponym, df):
    df.append({"hypernym": hypernym, "hyponym": hyponym})
    return df

def extract_hearst_patterns(result,df, parent_asin):
    for line in result.split("\n"):
        if "hyponym" in line:
            # extract content inside the parentheses
            match = re.search(r'hyponym\("([^"]+)"\s*;\s*"([^"]+)"\)', line) 
            if match:
                hypernym, hyponym = match.groups()
                # Use pandas concat instead of append which is deprecated
                df = pd.concat([df, pd.DataFrame([{"parent_asin":parent_asin,"hypernym": hypernym, "hyponym": hyponym}])], ignore_index=True)
    return df


        
# try with iloc 1:10
for i in range(len(metadata_filtered_df.iloc[0:10])):
    print(f"Processing row {i+1} of {len(metadata_filtered_df)}")
    parent_asin = metadata_filtered_df.iloc[i].parent_asin
    result = chain(prompt, model, metadata_filtered_df.iloc[i].source_text)
    print(result)
    print("--------------------------------")
    # Pass hearst_df as an argument to extract_hearst_patterns to avoid UnboundLocalError
    hearst_df = extract_hearst_patterns(result,hearst_df, parent_asin)


Processing row 1 of 324
hyponym("Oil"; "Organic Sweet Almond Oil and Fractionated Coconut Oil Bundle for Hair and Skin"), 
hyponym("Bundle"; "Organic Sweet Almond Oil and Fractionated Coconut Oil Bundle for Hair and Skin"), 
hyponym("Brand"; "Shiny Leaf"), 
hyponym("Scent"; "Almond Oil and Fractionated Coconut"), 
hyponym("Item Form"; "Oil"), 
hyponym("Unit Count"; "32.00 Fl Oz"), 
hyponym("Number of Items"; "2"), 
hyponym("Package Dimensions"; "8.62 x 5.04 x 2.48 inches; 2.23 Pounds"), 
hyponym("UPC"; "781584950669"), 
hyponym("Store"; "Shiny Leaf"), 
hyponym("Categories"; "*"), 
hyponym("Price"; "None")
--------------------------------
Processing row 2 of 324
hyponym("Glass Bottle"; "Empty Brown Glass Spray Bottles2-Pack"), hyponym("Brand"; "Cherry"), hyponym("Material"; "Glass"), hyponym("Capacity"; "16 Ounces"), hyponym("Number of Items"; "2"), hyponym("Product Care Instructions"; "Hand Wash Only"), hyponym("Package Dimensions"; "9.09 x 6.46 x 3.35 inches"), hyponym("UPC"; "7915232